In [ ]:
# 1) Download the dataset with the Kaggle CLI (command from the dataset page) and unzip it
!kaggle datasets download jacksaleeby/nasdaq100-historical-data-2000-2026-upvote -p /content/data --unzip
!ls -lh /content/data

Dataset URL: https://www.kaggle.com/datasets/jacksaleeby/nasdaq100-historical-data-2000-2026-upvote
License(s): CC-BY-SA-4.0
100% 8.02M/8.02M [00:00<00:00, 10.6MB/s]

total 27M
-rw-r--r-- 1 root root 27M Nov  7 19:58 NASDAQ100_Historical_Data.csv


In [ ]:
# 2) Data check: numbers for the Data section of proposal.txt
DATA_DIR = "/content/data"
import glob
import numpy as np
import pandas as pd

csv_path = glob.glob(DATA_DIR + "/**/NASDAQ100_Historical_Data.csv", recursive=True)[0]
df = pd.read_csv(csv_path, parse_dates=["Date"])
cols = list(df.columns)

N, ncol = df.shape
k = df["Ticker"].nunique()
d0, d1 = df["Date"].min().date(), df["Date"].max().date()
m = int(df.isna().sum().sum())
dups = int(df.duplicated(["Ticker", "Date"]).sum())
print(f"file: {csv_path}")
print(f"rows={N:,}  cols={ncol}  tickers={k}  dates={d0} .. {d1}  missing={m}  duplicate ticker-dates={dups}")
if m:
    print(df.isna().sum()[df.isna().sum() > 0])

# next-day target and 200-day warm-up (SMA-200 needs 200 rows per ticker)
df = df.sort_values(["Ticker", "Date"]).reset_index(drop=True)
nxt = df.groupby("Ticker")["Adj Close"].shift(-1)
df["target"] = np.where(nxt.notna(), (nxt > df["Adj Close"]).astype(float), np.nan)
warm = df.groupby("Ticker").cumcount() >= 199
usable = df[warm & df["target"].notna()]
N2 = len(usable)
p = round(100 * usable["target"].mean(), 1)
ties = round(100 * (nxt[usable.index] == usable["Adj Close"]).mean(), 1)
print(f"usable stock-days={N2:,}  positive={p}%  zero-change days={ties}%")

print("\n--- paste these two lines into the Data section of proposal.txt ---")
print(f"Loaded: {N:,} rows, {ncol} columns ({', '.join(cols)}), {k} tickers, {d0} to {d1}, {m} missing values.")
print(f"We compute the competition's 27 documented indicators per ticker from past prices only (27 inputs); "
      f"target = 1 if next-day Adj Close is higher. After a 200-day warm-up: {N2:,} stock-days, {p}% positive.")

file: /content/data/NASDAQ100_Historical_Data.csv
rows=514,075  cols=8  tickers=100  dates=2000-01-03 .. 2026-02-18  missing=0  duplicate ticker-dates=0
usable stock-days=494,075  positive=50.4%  zero-change days=2.9%

--- paste these two lines into the Data section of proposal.txt ---
Loaded: 514,075 rows, 8 columns (Ticker, Date, Open, High, Low, Close, Adj Close, Volume), 100 tickers, 2000-01-03 to 2026-02-18, 0 missing values.
We compute the competition's 27 documented indicators per ticker from past prices only (27 inputs); target = 1 if next-day Adj Close is higher. After a 200-day warm-up: 494,075 stock-days, 50.4% positive.


In [ ]:
print(df)

       Ticker       Date    Open    High     Low   Close  Adj Close  \
0        AAPL 2000-01-03    0.94    1.00    0.91    1.00       0.84   
1        AAPL 2000-01-04    0.97    0.99    0.90    0.92       0.77   
2        AAPL 2000-01-05    0.93    0.99    0.92    0.93       0.78   
3        AAPL 2000-01-06    0.95    0.96    0.85    0.85       0.71   
4        AAPL 2000-01-07    0.86    0.90    0.85    0.89       0.75   
...       ...        ...     ...     ...     ...     ...        ...   
514070     ZS 2026-02-11  175.62  175.87  165.36  171.00     171.00   
514071     ZS 2026-02-12  171.99  173.99  163.00  170.90     170.90   
514072     ZS 2026-02-13  174.34  179.90  172.43  177.72     177.72   
514073     ZS 2026-02-17  175.82  176.85  169.61  172.59     172.59   
514074     ZS 2026-02-18  167.35  172.65  164.39  172.13     172.13   

           Volume  target  
0       535796800     0.0  
1       512377600     1.0  
2       778321600     0.0  
3       767972800     1.0  
4      

In [ ]:
# @title 0. Reproducible project setup (run after the original data cells)
%pip -q install lightgbm==4.6.0
import os, sys, json, time, hashlib, platform, shutil, warnings, subprocess
from pathlib import Path
import numpy as np
import pandas as pd
import scipy, sklearn, lightgbm
import matplotlib.pyplot as plt
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import KFold
from sklearn.exceptions import ConvergenceWarning
from lightgbm import LGBMClassifier
from threadpoolctl import threadpool_limits
from IPython.display import display, Markdown
SEED = 5420
N_BOOT = 1000
TEST_YEARS = list(range(2012, 2024))
SCHEMES = ['shuffled', 'date_grouped', 'walk_forward']
WORK = Path('/content/ese5420_cv_project')
WORK.mkdir(parents=True, exist_ok=True)
DRIVE_PROJECT = Path('/content/drive/MyDrive/ESE5420_CV_Project')
OUT = DRIVE_PROJECT if Path('/content/drive/MyDrive').exists() else WORK / 'outputs'
for part in ['', 'audit', 'predictions', 'cache', 'tables', 'figures', 'reports']:
    (OUT / part).mkdir(parents=True, exist_ok=True)
CSV = Path('/content/data/NASDAQ100_Historical_Data.csv')
versions = {k: v for k, v in [('python',sys.version), ('numpy',np.__version__), ('pandas',pd.__version__), ('scipy',scipy.__version__), ('sklearn',sklearn.__version__), ('lightgbm',lightgbm.__version__)]}
(OUT / 'audit' / 'environment.json').write_text(json.dumps(versions, indent=2))
print('Setup complete. CPUs:', os.cpu_count(), '| data exists:', CSV.exists())
print('Output directory:', OUT)
print('Drive mounted:', Path('/content/drive/MyDrive').exists())
print('Annual experiments: 2012-2023. Final 2024-2026 performance remains unopened.')


Setup complete. CPUs: 2 | data exists: False
Output directory: /content/ese5420_cv_project/outputs
Drive mounted: False
Annual experiments: 2012-2023. Final 2024-2026 performance remains unopened.


In [ ]:
# @title 1. Persistent project storage in Google Drive
from google.colab import drive
drive.mount('/content/drive')
previous_out = OUT
OUT = Path('/content/drive/MyDrive/ESE5420_CV_Project')
OUT.mkdir(parents=True, exist_ok=True)
if previous_out != OUT and previous_out.exists():
    shutil.copytree(previous_out, OUT, dirs_exist_ok=True)
for part in ['audit', 'predictions', 'cache', 'tables', 'figures', 'reports']:
    (OUT / part).mkdir(parents=True, exist_ok=True)
(OUT / 'audit' / 'environment.json').write_text(json.dumps(versions, indent=2))
print('Persistent checkpoint directory:', OUT)


Mounted at /content/drive
Persistent checkpoint directory: /content/drive/MyDrive/ESE5420_CV_Project


In [ ]:
# @title 2. Data audit and 27 explicitly defined, causal features
DATASET_URL = 'https://www.kaggle.com/datasets/jacksaleeby/nasdaq100-historical-data-2000-2026-upvote'
assert CSV.exists(), 'Run the original Kaggle download cell first.'
raw = pd.read_csv(CSV, parse_dates=['Date']).sort_values(['Ticker','Date']).reset_index(drop=True)
assert len(raw) == 514075 and raw.Ticker.nunique() == 100, 'Dataset changed: review before using the registered protocol.'
assert not raw.duplicated(['Ticker','Date']).any()
assert not raw.isna().any().any()
assert (raw[['Open','High','Low','Close','Adj Close']] > 0).all().all()
DATA_SHA = hashlib.sha256(CSV.read_bytes()).hexdigest()
FEATURE_VERSION = 'causal27_v1'
feature_source = r'''
def one_ticker_features(g):
    g = g.sort_values('Date').copy()
    c = g['Adj Close'].astype(float)
    factor = c / g['Close']
    o, h, l = g['Open']*factor, g['High']*factor, g['Low']*factor
    v = g['Volume'].astype(float)
    f = pd.DataFrame(index=g.index)
    for n in [1,2,5,10,20]:
        f[f'return_{n}'] = c.pct_change(n, fill_method=None)
    for n in [5,10,20,50,200]:
        f[f'sma_ratio_{n}'] = c/c.rolling(n, min_periods=n).mean()-1
    e12 = c.ewm(span=12, adjust=False, min_periods=12).mean()
    e26 = c.ewm(span=26, adjust=False, min_periods=26).mean()
    f['ema_ratio_12'], f['ema_ratio_26'] = c/e12-1, c/e26-1
    ret = c.pct_change(fill_method=None)
    for n in [5,20,60]:
        f[f'volatility_{n}'] = ret.rolling(n, min_periods=n).std(ddof=1)
    delta = c.diff()
    gain = delta.clip(lower=0).ewm(alpha=1/14, adjust=False, min_periods=14).mean()
    loss = (-delta.clip(upper=0)).ewm(alpha=1/14, adjust=False, min_periods=14).mean()
    rsi = gain/(gain+loss)
    f['rsi_14'] = rsi.mask((gain+loss)==0, 0.5)
    macd = e12-e26
    signal = macd.ewm(span=9, adjust=False, min_periods=9).mean()
    f['macd_relative'], f['macd_signal_relative'], f['macd_hist_relative'] = macd/c, signal/c, (macd-signal)/c
    sma20 = c.rolling(20).mean()
    sd20 = c.rolling(20).std(ddof=1)
    f['bollinger_z_20'] = (c-sma20)/sd20.replace(0,np.nan)
    f['bollinger_width_20'] = 4*sd20/sma20
    tr = pd.concat([h-l, (h-c.shift()).abs(), (l-c.shift()).abs()], axis=1).max(axis=1)
    f['atr_relative_14'] = tr.ewm(alpha=1/14, adjust=False, min_periods=14).mean()/c
    hi, lo = h.rolling(14).max(), l.rolling(14).min()
    k = (c-lo)/(hi-lo).replace(0,np.nan)
    k = k.mask((hi-lo)==0, 0.5)
    f['stochastic_k_14'], f['stochastic_d_3'] = k, k.rolling(3).mean()
    f['volume_relative_20'] = v/v.rolling(20).mean().replace(0,np.nan)-1
    f['intraday_return'] = c/o-1
    f['overnight_gap'] = o/c.shift()-1
    return f.replace([np.inf,-np.inf],np.nan)

def build_feature_frame(data):
    return pd.concat([one_ticker_features(g) for _,g in data.groupby('Ticker',sort=False)]).sort_index()
'''
exec(feature_source)
(OUT/'audit'/'feature_implementation.py').write_text(feature_source)
FEATURES = list(one_ticker_features(raw[raw.Ticker == raw.Ticker.iloc[0]]).columns)
assert len(FEATURES) == 27
feature_frame = build_feature_frame(raw)
calendar = np.sort(raw.Date.unique())
next_market_date = pd.Series(calendar[1:], index=calendar[:-1])
label_date = raw.groupby('Ticker').Date.shift(-1)
next_close = raw.groupby('Ticker')['Adj Close'].shift(-1)
valid_horizon = label_date.eq(raw.Date.map(next_market_date))
next_return = next_close/raw['Adj Close']-1
label = (next_return > 0).astype('int8')
median_return = next_return.where(valid_horizon).groupby(raw.Date).transform('median')
metadata = raw[['Ticker','Date','Close']].copy()
metadata['LabelDate'] = label_date
metadata['y'] = label
metadata['y_median'] = (next_return > median_return).astype('int8')
metadata['row_id'] = np.arange(len(raw))
mask = (raw.groupby('Ticker').cumcount() >= 199) & next_close.notna() & valid_horizon
panel = pd.concat([metadata,feature_frame],axis=1).loc[mask].sort_values(['Date','Ticker']).reset_index(drop=True)
panel['Ticker'] = panel.Ticker.astype('category')
assert panel[FEATURES].notna().any(axis=0).all()
assert (panel.LabelDate > panel.Date).all()
# Core analyses never use outcomes on/after 2024-01-01.
pre_final = panel[(panel.Date < '2024-01-01') & (panel.LabelDate < '2024-01-01')]
audit = {'raw_rows':len(raw),'tickers':int(raw.Ticker.nunique()),'start':str(raw.Date.min().date()),'end':str(raw.Date.max().date()),'source_url':DATASET_URL,'data_sha256':DATA_SHA,'feature_version':FEATURE_VERSION,'feature_count':len(FEATURES),'usable_rows':len(panel),'non_next_market_day_rows_excluded':int((next_close.notna() & ~valid_horizon).sum()),'pre_final_positive_rate':float(pre_final.y.mean()),'feature_missing_cells':int(panel[FEATURES].isna().sum().sum()),'license':'CC BY-SA 4.0','upstream_source':'not named by dataset card','constituents':'current constituents backfilled; survivorship bias','feature_provenance':'project-defined, NOT competition-provided; approved by Yixun'}
(OUT/'audit'/'data_audit.json').write_text(json.dumps(audit,indent=2))
# A private reproducibility copy; never included in the public aggregate report bundle.
(OUT/'private_data').mkdir(exist_ok=True)
shutil.copy2(CSV, OUT/'private_data'/CSV.name)
panel.to_parquet(OUT/'private_data'/'features.parquet',index=False)
annual_counts = pre_final.groupby(pre_final.Date.dt.year).agg(stock_days=('y','size'),trading_days=('Date','nunique'),tickers=('Ticker','nunique'),positive_rate=('y','mean'))
annual_counts.to_csv(OUT/'tables'/'development_data_by_year.csv')
print(json.dumps(audit,indent=2))
display(annual_counts.loc[2012:2023])
print('27 features:', ', '.join(FEATURES))


{
  "raw_rows": 514075,
  "tickers": 100,
  "start": "2000-01-03",
  "end": "2026-02-18",
  "source_url": "https://www.kaggle.com/datasets/jacksaleeby/nasdaq100-historical-data-2000-2026-upvote",
  "data_sha256": "5355285b81706b2feeadda5cda4492ec17c61fec3b9b0bd4adcc8aab1966a286",
  "feature_version": "causal27_v1",
  "feature_count": 27,
  "usable_rows": 494075,
  "non_next_market_day_rows_excluded": 0,
  "pre_final_positive_rate": 0.5027654264972776,
  "feature_missing_cells": 398,
  "license": "CC BY-SA 4.0",
  "upstream_source": "not named by dataset card",
  "constituents": "current constituents backfilled; survivorship bias",
  "feature_provenance": "project-defined, NOT competition-provided; approved by Yixun"
}


,stock_days,trading_days,tickers,positive_rate
Date,,,,
2012,19000,250,76,0.487737
2013,19737,252,80,0.515529
2014,20343,252,81,0.516689
2015,20412,252,81,0.498726
2016,20839,252,84,0.519651
2017,21207,251,85,0.537794
2018,21438,251,87,0.515766
2019,22155,252,89,0.547596
2020,22838,253,91,0.535336


27 features: return_1, return_2, return_5, return_10, return_20, sma_ratio_5, sma_ratio_10, sma_ratio_20, sma_ratio_50, sma_ratio_200, ema_ratio_12, ema_ratio_26, volatility_5, volatility_20, volatility_60, rsi_14, macd_relative, macd_signal_relative, macd_hist_relative, bollinger_z_20, bollinger_width_20, atr_relative_14, stochastic_k_14, stochastic_d_3, volume_relative_20, intraday_return, overnight_gap


In [ ]:
# @title 3. Leakage tests, validation splits, and frozen experiment protocol
formula_rows = []
for n in [1,2,5,10,20]: formula_rows.append((f'return_{n}',f'AdjustedClose[t]/AdjustedClose[t-{n}]-1'))
for n in [5,10,20,50,200]: formula_rows.append((f'sma_ratio_{n}',f'AdjustedClose / trailing SMA({n}) - 1'))
for n in [12,26]: formula_rows.append((f'ema_ratio_{n}',f'AdjustedClose / EMA(span={n}, adjust=False) - 1'))
for n in [5,20,60]: formula_rows.append((f'volatility_{n}',f'Trailing sample standard deviation of {n} one-day simple returns'))
formula_rows += [('rsi_14','EWMA(gain,alpha=1/14) / [EWMA(gain)+EWMA(loss)]; flat=0.5; min_periods=14'),('macd_relative','[EMA12-EMA26]/AdjustedClose'),('macd_signal_relative','EMA9(MACD)/AdjustedClose'),('macd_hist_relative','[MACD-EMA9(MACD)]/AdjustedClose'),('bollinger_z_20','[AdjustedClose-SMA20]/sample_std20'),('bollinger_width_20','4*sample_std20/SMA20'),('atr_relative_14','EWMA(true_range,alpha=1/14)/AdjustedClose'),('stochastic_k_14','[AdjustedClose-low14]/[high14-low14], flat=0.5'),('stochastic_d_3','Trailing mean of stochastic_k_14 over 3 observations'),('volume_relative_20','Volume/trailing_SMA20(Volume)-1'),('intraday_return','AdjustedClose/AdjustedOpen-1'),('overnight_gap','AdjustedOpen[t]/AdjustedClose[t-1]-1')]
pd.DataFrame(formula_rows,columns=['feature','formula']).to_csv(OUT/'audit'/'feature_dictionary.csv',index=False)
assert [r[0] for r in formula_rows] == FEATURES
leakage_results = []
for cutoff in ['2006-12-29','2014-12-31','2022-12-30']:
    altered = raw.copy()
    future = altered.Date > cutoff
    perturb_rng = np.random.default_rng(SEED)
    multipliers = perturb_rng.uniform(0.5,2.0,size=int(future.sum()))
    altered.loc[future,['Open','High','Low','Close','Adj Close']] = altered.loc[future,['Open','High','Low','Close','Adj Close']].mul(multipliers,axis=0)
    altered.loc[future,'Volume'] = np.round(altered.loc[future,'Volume'].to_numpy()*multipliers).astype('int64')
    check_features = build_feature_frame(altered)
    np.testing.assert_allclose(feature_frame.loc[~future,FEATURES],check_features.loc[~future,FEATURES],rtol=0,atol=1e-12,equal_nan=True)
    leakage_results.append({'cutoff':cutoff,'past_rows_checked':int((~future).sum()),'future_rows_changed':int(future.sum()),'status':'PASS'})
    print('Future-perturbation test PASS:',cutoff,flush=True)
del altered, check_features
(OUT/'audit'/'feature_leakage_tests.json').write_text(json.dumps(leakage_results,indent=2))

def get_variant(name='main'):
    p = panel.copy()
    if name == 'close_ge_5': p = p.loc[p.Close >= 5].copy()
    elif name == 'beats_daily_median': p['y'] = p['y_median']
    elif name != 'main': raise ValueError(name)
    return p.reset_index(drop=True)

def annual_data(year, variant='main', final=False):
    p = get_variant(variant)
    start = pd.Timestamp(year,1,1)
    end = pd.Timestamp(year+1,1,1)
    dev = p.loc[(p.Date < start) & (p.LabelDate < start)].reset_index(drop=True)
    test_mask = (p.Date >= start) & (p.Date < end) & (p.LabelDate < end)
    if final:
        assert year == 2024
        test_mask = (p.Date >= start)
    test = p.loc[test_mask].reset_index(drop=True)
    assert len(dev)>0 and len(test)>0 and dev.LabelDate.max() < test.Date.min()
    return dev,test

def make_splits(dev, year, scheme):
    dates = dev.Date.to_numpy()
    label_dates = dev.LabelDate.to_numpy()
    if scheme == 'shuffled':
        splits = list(KFold(5,shuffle=True,random_state=SEED+year).split(dev))
    elif scheme == 'date_grouped':
        days, day_id = np.unique(dates,return_inverse=True)
        splits = []
        for tr_days,va_days in KFold(5,shuffle=True,random_state=SEED+year).split(days):
            va = np.flatnonzero(np.isin(day_id,va_days))
            tr = np.flatnonzero(~np.isin(day_id,va_days))
            assert len(np.intersect1d(dates[tr],dates[va]))==0
            splits.append((tr,va))
    elif scheme == 'walk_forward':
        splits = []
        for val_year in range(year-5,year):
            start,end = np.datetime64(f'{val_year}-01-01'), np.datetime64(f'{val_year+1}-01-01')
            va = np.flatnonzero((dates>=start)&(dates<end)&(label_dates<end))
            assert len(va)>0
            tr = np.flatnonzero((dates<dates[va].min()) & (label_dates<dates[va].min()))
            assert len(tr)>0 and label_dates[tr].max()<dates[va].min()
            splits.append((tr,va))
    else: raise ValueError(scheme)
    for tr,va in splits:
        assert len(np.intersect1d(tr,va))==0
        assert dev.iloc[tr].y.nunique()==2 and dev.iloc[va].y.nunique()==2
    return splits

GRIDS = {
 'logistic': [{'C':float(c)} for c in np.logspace(-3,1,5)],
 'lightgbm': [
  {'n_estimators':75,'num_leaves':7,'min_child_samples':200},
  {'n_estimators':100,'num_leaves':15,'min_child_samples':200},
  {'n_estimators':150,'num_leaves':15,'min_child_samples':200},
  {'n_estimators':100,'num_leaves':31,'min_child_samples':200},
  {'n_estimators':150,'num_leaves':31,'min_child_samples':400}]}
DEFAULT_IDS = {'logistic':3, 'lightgbm':1}

def make_model(name, config_id, year):
    params = GRIDS[name][config_id]
    if name == 'logistic':
        est = LogisticRegression(penalty='l2',solver='newton-cholesky',max_iter=200,tol=1e-6,random_state=SEED+year,**params)
        return make_pipeline(SimpleImputer(strategy='median'),StandardScaler(),est)
    if name == 'lightgbm':
        est = LGBMClassifier(objective='binary',learning_rate=0.05,n_jobs=2,verbosity=-1,random_state=SEED+year,deterministic=True,force_col_wise=True,**params)
        return make_pipeline(SimpleImputer(strategy='median'),est)
    raise ValueError(name)

protocol = {'version':'2026-11-08-v1','seed':SEED,'bootstrap_resamples':N_BOOT,'years':TEST_YEARS,'schemes':SCHEMES,'features':FEATURES,'feature_sha256':hashlib.sha256(feature_source.encode()).hexdigest(),'data_sha256':DATA_SHA,'grids':GRIDS,'default_ids':DEFAULT_IDS,'cv_aggregation':'unweighted mean of five fold AUCs','date_grouping':'randomly shuffled unique dates split into five folds','walk_forward':'five last development calendar years; expanding training; train LabelDate strictly before first validation Date','annual_boundary':'both feature date and outcome date must fall before next test year boundary','final_test':'2024-01-01 through last labeled observation in February 2026; accessed after protocol lock','selection':'candidate pipelines fitted separately inside development CV; no year-Y outcomes used to select; selected CV score not an unbiased performance estimate','uncertainty':'day bootstrap conditional on fitted model; paired year bootstrap exploratory because development sets overlap','interpretation':'shuffled/date-grouped/walk-forward contrasts also differ in validation periods and training sizes; not a pure causal decomposition of leakage'}
PROTOCOL_ID = hashlib.sha256(json.dumps(protocol,sort_keys=True).encode()).hexdigest()[:12]
(OUT/'audit'/'protocol.json').write_text(json.dumps(protocol,indent=2))
(OUT/'audit'/'protocol_id.txt').write_text(PROTOCOL_ID)
(OUT/'cache'/PROTOCOL_ID).mkdir(exist_ok=True)
split_audit = []
for yr in TEST_YEARS:
    dev,test = annual_data(yr)
    for scheme in SCHEMES:
        for fold,(tr,va) in enumerate(make_splits(dev,yr,scheme)):
            split_audit.append({'test_year':yr,'scheme':scheme,'fold':fold,'n_train':len(tr),'n_valid':len(va),'train_end':str(dev.iloc[tr].Date.max().date()),'train_label_end':str(dev.iloc[tr].LabelDate.max().date()),'valid_start':str(dev.iloc[va].Date.min().date()),'valid_end':str(dev.iloc[va].Date.max().date()),'test_n':len(test)})
pd.DataFrame(split_audit).to_csv(OUT/'audit'/'split_audit.csv',index=False)
print('PASS: 180 audited splits; protocol:',PROTOCOL_ID)
print('No 2024-2026 model performance has been computed.')


Future-perturbation test PASS: 2006-12-29
Future-perturbation test PASS: 2014-12-31
Future-perturbation test PASS: 2022-12-30


In [ ]:
# @title 4. Cached training engine and trading-day bootstrap
CACHE = OUT/'cache'/PROTOCOL_ID
PRED = OUT/'predictions'/PROTOCOL_ID
PRED.mkdir(exist_ok=True)

def atomic_json(path,obj):
    tmp = Path(str(path)+'.tmp')
    tmp.write_text(json.dumps(obj,indent=2,default=str))
    tmp.replace(path)

class WeightedAUC:
    # Sort predictions once; retain exact tie handling when date weights change.
    def __init__(self,y,score):
        y,score = np.asarray(y,dtype=float),np.asarray(score,dtype=float)
        assert np.isfinite(score).all() and set(np.unique(y)) <= {0.,1.}
        self.order = np.argsort(score,kind='stable')
        self.y = y[self.order]
        s = score[self.order]
        self.starts = np.r_[0,np.flatnonzero(np.diff(s)!=0)+1]
    def __call__(self,weights):
        w = np.asarray(weights,dtype=float)[self.order]
        pos = np.add.reduceat(w*self.y,self.starts)
        neg = np.add.reduceat(w*(1-self.y),self.starts)
        denom = pos.sum()*neg.sum()
        return float(np.sum(pos*(np.cumsum(neg)-0.5*neg))/denom) if denom>0 else np.nan

def day_bootstrap(dates,y,score,n_boot=N_BOOT,seed=SEED,block_days=1):
    days,ids = np.unique(np.asarray(dates),return_inverse=True)
    scorer = WeightedAUC(y,score)
    rng = np.random.default_rng(seed)
    boot = np.empty(n_boot)
    for b in range(n_boot):
        if block_days==1:
            sampled = rng.integers(0,len(days),size=len(days))
        else:
            starts = rng.integers(0,len(days),size=int(np.ceil(len(days)/block_days)))
            sampled = ((starts[:,None]+np.arange(block_days))%len(days)).ravel()[:len(days)]
        counts = np.bincount(sampled,minlength=len(days))
        boot[b] = scorer(counts[ids])
    assert np.isfinite(boot).all(), 'Degenerate class bootstrap sample: inspect explicitly.'
    return boot

# Independent check against sklearn, including ties, zero weights, and repeated days.
rng_check = np.random.default_rng(SEED)
for n in [20,101,1000]:
    yy = rng_check.integers(0,2,n)
    ss = np.round(rng_check.normal(size=n),1)
    ww = rng_check.integers(0,4,n)
    np.testing.assert_allclose(WeightedAUC(yy,ss)(ww),roc_auc_score(yy,ss,sample_weight=ww),rtol=0,atol=1e-12)
print('PASS: bootstrap AUC agrees with sklearn on independent weighted/tied-score cases.')

META = ['row_id','Ticker','Date','LabelDate','y']
def fit_predict(name,cid,year,xtrain,ytrain,xtest):
    est = make_model(name,cid,year)
    began = time.perf_counter()
    with threadpool_limits(limits=2), warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter('always',ConvergenceWarning)
        warnings.filterwarnings('ignore',message='X does not have valid feature names')
        est.fit(xtrain,ytrain)
        score = est.predict_proba(xtest)[:,1]
    failures = [str(w.message) for w in caught if issubclass(w.category,ConvergenceWarning)]
    if failures: raise RuntimeError('Convergence warning: '+str(failures))
    assert np.isfinite(score).all()
    return score,float(time.perf_counter()-began)

def cv_run(dev,year,scheme,name,cid,variant='main',save_predictions=False,force=False):
    stem = f'{variant}_{year}_{scheme}_{name}_c{cid}'
    path,predpath = CACHE/(stem+'_cv.json'), PRED/(stem+'_cv.parquet')
    if path.exists() and not force and (not save_predictions or predpath.exists()):
        return json.loads(path.read_text())
    x = dev[FEATURES].to_numpy(dtype=np.float64)
    y = dev.y.to_numpy(dtype=np.int8)
    folds,predictions = [],[]
    for fold,(tr,va) in enumerate(make_splits(dev,year,scheme)):
        scores,seconds = fit_predict(name,cid,year,x[tr],y[tr],x[va])
        auc = float(roc_auc_score(y[va],scores))
        folds.append({'fold':fold,'auc':auc,'n_train':len(tr),'n_valid':len(va),'seconds':seconds})
        if save_predictions:
            q = dev.iloc[va][META].copy()
            q['score'],q['fold'] = scores,fold
            predictions.append(q)
    result = {'variant':variant,'year':int(year),'scheme':scheme,'model':name,'config_id':int(cid),'params':GRIDS[name][cid],'cv_auc':float(np.mean([f['auc'] for f in folds])),'folds':folds,'seconds':float(sum(f['seconds'] for f in folds)),'protocol_id':PROTOCOL_ID}
    if save_predictions: pd.concat(predictions,ignore_index=True).to_parquet(predpath,index=False)
    atomic_json(path,result)
    return result

def test_run(dev,test,year,name,cid,variant='main',final=False,force=False):
    prefix = 'final' if final else 'annual'
    stem = f'{prefix}_{variant}_{year}_{name}_c{cid}'
    path,predpath,bootpath = CACHE/(stem+'_test.json'),PRED/(stem+'_test.parquet'),CACHE/(stem+'_boot.npy')
    if path.exists() and predpath.exists() and bootpath.exists() and not force:
        return json.loads(path.read_text())
    score,seconds = fit_predict(name,cid,year,dev[FEATURES].to_numpy(dtype=np.float64),dev.y.to_numpy(),test[FEATURES].to_numpy(dtype=np.float64))
    q = test[META].copy()
    q['score'] = score
    q.to_parquet(predpath,index=False)
    auc = float(roc_auc_score(test.y,score))
    boots = day_bootstrap(test.Date.to_numpy(),test.y.to_numpy(),score,seed=SEED+year)
    np.save(bootpath,boots)
    lo,hi = np.quantile(boots,[0.025,0.975])
    result = {'variant':variant,'year':int(year),'model':name,'config_id':int(cid),'test_auc':auc,'test_lo':float(lo),'test_hi':float(hi),'n_test':len(test),'n_days':int(test.Date.nunique()),'n_train':len(dev),'fit_seconds':seconds,'prediction_file':str(predpath),'bootstrap_file':str(bootpath),'final':bool(final),'protocol_id':PROTOCOL_ID}
    atomic_json(path,result)
    return result

def run_fixed(years=TEST_YEARS,models=('logistic','lightgbm'),variants=('main',)):
    completed=[]
    for variant in variants:
        for year in years:
            dev,test = annual_data(year,variant)
            for name in models:
                cid = DEFAULT_IDS[name]
                truth = test_run(dev,test,year,name,cid,variant)
                for scheme in SCHEMES:
                    estimate = cv_run(dev,year,scheme,name,cid,variant,save_predictions=True)
                    row = {k:truth[k] for k in ['year','variant','model','config_id','test_auc','test_lo','test_hi','n_test','n_days','n_train']}
                    row.update(scheme=scheme,cv_auc=estimate['cv_auc'],error=estimate['cv_auc']-truth['test_auc'],error_test_only_lo=estimate['cv_auc']-truth['test_hi'],error_test_only_hi=estimate['cv_auc']-truth['test_lo'],protocol_id=PROTOCOL_ID)
                    atomic_json(CACHE/f'fixed_{variant}_{year}_{name}_{scheme}.json',row)
                    completed.append(row)
                    print(f'{variant} | {year} | {name:8s} | {scheme:12s} | CV={row["cv_auc"]:.5f} test={row["test_auc"]:.5f} error={row["error"]:+.5f}',flush=True)
            # An interruption loses at most the currently running candidate/folds.
            gather_fixed()
    return pd.DataFrame(completed)

def gather_fixed():
    files = sorted(CACHE.glob('fixed_*.json'))
    result = pd.DataFrame([json.loads(p.read_text()) for p in files])
    if len(result):
        result = result.sort_values(['variant','year','model','scheme']).reset_index(drop=True)
        result.to_csv(OUT/'tables'/'fixed_bias_by_year.csv',index=False)
    return result

atomic_json(OUT/'audit'/'bootstrap_checks.json',{'status':'PASS','reference':'sklearn.metrics.roc_auc_score','tests':'random tied-score weighted comparisons','resampling_unit':'whole trading day, with replacement','n_resamples':N_BOOT,'seed':SEED})
print('Engine ready. All fits use fold-specific preprocessing; cached results are keyed by the frozen protocol.')


In [ ]:
# @title 5. Baseline checkpoint: one test year, both core models
pilot_results = run_fixed(years=[2012])
display(pilot_results[['year','model','scheme','cv_auc','test_auc','test_lo','test_hi','error']])
print('Pilot complete. Outputs are checkpointed in Google Drive.')


In [1]:
import pandas, sklearn, lightgbm
print(pandas.__version__, sklearn.__version__, lightgbm.__version__)

OSError: dlopen(/opt/anaconda3/envs/finalpaper/lib/python3.11/site-packages/lightgbm/lib/lib_lightgbm.dylib, 0x0006): Library not loaded: @rpath/libomp.dylib
  Referenced from: <59CCDAC2-EBAE-32F7-ADF4-469B834532C9> /opt/anaconda3/envs/finalpaper/lib/python3.11/site-packages/lightgbm/lib/lib_lightgbm.dylib
  Reason: tried: '/opt/homebrew/opt/libomp/lib/libomp.dylib' (no such file), '/System/Volumes/Preboot/Cryptexes/OS/opt/homebrew/opt/libomp/lib/libomp.dylib' (no such file), '/opt/local/lib/libomp/libomp.dylib' (no such file), '/System/Volumes/Preboot/Cryptexes/OS/opt/local/lib/libomp/libomp.dylib' (no such file), '/opt/homebrew/opt/libomp/lib/libomp.dylib' (no such file), '/System/Volumes/Preboot/Cryptexes/OS/opt/homebrew/opt/libomp/lib/libomp.dylib' (no such file), '/opt/local/lib/libomp/libomp.dylib' (no such file), '/System/Volumes/Preboot/Cryptexes/OS/opt/local/lib/libomp/libomp.dylib' (no such file), '/opt/anaconda3/envs/finalpaper/lib/python3.11/lib-dynload/../../libomp.dylib' 